In [1]:
from datasets import load_dataset

dataset = load_dataset("ButterChicken98/plantvillage-image-text-pairs")

dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'caption', 'captions'],
        num_rows: 20638
    })
})

In [2]:
import pandas as pd

df = dataset["train"].to_pandas()

df.head()

,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."


In [3]:
df.to_csv("dataset_raw.csv", index=False)

print("CSV created!")

CSV created!


In [4]:
# Expand caption list into multiple rows
df = df.explode("captions", ignore_index=True)

# Rename captions column → text
df = df.rename(columns={"captions": "text"})

df.head()

,image,caption,text
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,A vibrant green and healthy tomato leaf with s...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,"A healthy Solanum lycopersicum leaf, free of d..."
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,"A fresh tomato leaf outdoors, glowing in sunli..."
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,"A clean and healthy tomato leaf image, perfect..."
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,A tomato leaf showing dark brown lesions and w...


In [5]:
def extract_label(text):
    text = text.lower()

    if "healthy" in text:
        return "healthy"
    elif "blight" in text:
        return "blight"
    elif "mildew" in text:
        return "mildew"
    elif "rust" in text:
        return "rust"
    elif "spot" in text:
        return "leaf_spot"
    else:
        return "other"

df["label_name"] = df["text"].apply(extract_label)

In [6]:
df = df[["label_name", "text"]]

df.head()

,label_name,text
0,healthy,A vibrant green and healthy tomato leaf with s...
1,healthy,"A healthy Solanum lycopersicum leaf, free of d..."
2,other,"A fresh tomato leaf outdoors, glowing in sunli..."
3,healthy,"A clean and healthy tomato leaf image, perfect..."
4,blight,A tomato leaf showing dark brown lesions and w...


In [7]:
df.to_csv("dataset.csv", index=False)

print("CSV created successfully")

CSV created successfully


In [8]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

import evaluate

In [9]:
df = pd.read_csv("dataset.csv")

df.head()

,label_name,text
0,healthy,A vibrant green and healthy tomato leaf with s...
1,healthy,"A healthy Solanum lycopersicum leaf, free of d..."
2,other,"A fresh tomato leaf outdoors, glowing in sunli..."
3,healthy,"A clean and healthy tomato leaf image, perfect..."
4,blight,A tomato leaf showing dark brown lesions and w...


In [19]:
encoder = LabelEncoder()

df["label"] = encoder.fit_transform(df["label_name"])

num_classes = len(encoder.classes_)

print("Number of classes:", num_classes)
print("Classes:", encoder.classes_)

Number of classes: 4
Classes: ['blight' 'healthy' 'leaf_spot' 'other']


In [11]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(df_train, preserve_index=False)
test_dataset = Dataset.from_pandas(df_test, preserve_index=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 66041
Test size: 16511


In [12]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [13]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_test = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/66041 [00:00<?, ? examples/s]

Map:   0%|          | 0/16511 [00:00<?, ? examples/s]

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [16]:
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

In [17]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [18]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [21]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_test = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/66041 [00:00<?, ? examples/s]

Map:   0%|          | 0/16511 [00:00<?, ? examples/s]

In [23]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.004088,0.000000,1.000000
2,0.000000,0.000000,1.000000
3,0.000000,0.000000,1.000000
4,0.000000,0.000000,1.000000
5,0.000000,0.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=20640, training_loss=0.0008175853086562932, metrics={'train_runtime': 2186.2466, 'train_samples_per_second': 151.037, 'train_steps_per_second': 9.441, 'total_flos': 2410656290733600.0, 'train_loss': 0.0008175853086562932, 'epoch': 5.0})

: 